In [2]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.document_loaders.csv_loader import CSVLoader

from markitdown import MarkItDown, StreamInfo
md = MarkItDown(enable_builtins=True)
result = md.convert("./tmp/movies.csv",stream_info=StreamInfo(charset="utf-8"))
with open("./tmp/movies.md","w",encoding="utf-8") as f:
    f.write(result.text_content)


loader = CSVLoader(file_path="./tmp/movies.csv", encoding="utf-8", metadata_columns=['id','title'], content_columns=['title','cast','director','overview','genres','keywords','tagline','release_date'])
documents = loader.load()

#configur embedding model
from langchain_huggingface import HuggingFaceEmbeddings
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m', model_kwargs={"device": device})


/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


In [2]:
!uv pip install -U faiss-cpu

Using Python 3.12.3 environment at: /home/admin/_github/massimodipaolo/ai-crash-course/.venv
Resolved 3 packages in 77ms                                          
Audited 3 packages in 0.13ms


In [25]:
from langchain_community.vectorstores import FAISS
import os, shutil
_storage_id=f"./tmp/db/{FAISS.__name__.lower()}"
if os.path.exists(_storage_id):
    shutil.rmtree(_storage_id, ignore_errors=True)
_db = FAISS.from_documents(documents, embeddings)
_db.save_local(_storage_id)
del _db

In [ ]:
!uv pip install -U langchain-chroma

In [3]:
from langchain_chroma import Chroma as CHROMA
import os, shutil
_storage_id = f"./tmp/db/{CHROMA.__name__.lower()}"
if os.path.exists(_storage_id):
	shutil.rmtree(_storage_id, ignore_errors=True)
_db = CHROMA.from_documents(documents=documents, embedding=embeddings, collection_name="movies", persist_directory=_storage_id)
del _db

In [ ]:
!uv pip install -U langchain-qdrant fastembed

In [27]:
from langchain_qdrant import QdrantVectorStore as QDRANT, FastEmbedSparse, RetrievalMode
import os, shutil
_storage_id=f"./tmp/db/qdrant"
if os.path.exists(_storage_id):
	shutil.rmtree(_storage_id, ignore_errors=True)
	
_db = QDRANT.from_documents(
                    documents=documents,
                    embedding=embeddings,
                    sparse_embedding=FastEmbedSparse(),
                    collection_name="movies",
                    path=_storage_id,
                    retrieval_mode=RetrievalMode.HYBRID)
_db.client.close()
del _db # prevent locking the database

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma as CHROMA
from langchain_qdrant import QdrantVectorStore as QDRANT, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient
#shared retrived parms
kwargs={"search_type":"similarity","search_kwargs":{"k":1}}
#faiss
faiss_db =FAISS.load_local(f"./tmp/db/{FAISS.__name__.lower()}", embeddings, allow_dangerous_deserialization=True)
query = "Who directed the movie Inception?"

#similarity search 
_ss = faiss_db.similarity_search(query, k=1, fetch_k=5)
print("similariry", [doc.metadata for doc in _ss])

"""
**MMR:**
- A **re-ranking strategy** applied after initial retrieval
- Balances **relevance AND diversity**
- Reduces redundancy by penalizing similar results
- Can works on top of any retrieval method (dense or sparse)

- `fetch_k=20`: Retrieves 20 candidates using similarity search
- `lambda_mult=0.5`: Balances relevance (1.0 = only relevance, 0.0 = only diversity)
- `k=1`: Returns 1 final result after re-ranking

MMR iteratively selects documents that are both relevant to the query AND dissimilar to already-selected documents, preventing duplicate or near-duplicate results.
"""
_mmr = faiss_db.max_marginal_relevance_search(query, k=1, fetch_k=20, lambda_mult=0.5)
print("mmr:", [doc.metadata for doc in _mmr])





#cleanup
#del faiss_db

similariry [{'source': './tmp/movies.csv', 'row': 96, 'id': '27205', 'title': 'Inception'}]
mmr: [{'source': './tmp/movies.csv', 'row': 96, 'id': '27205', 'title': 'Inception'}]


In [9]:
# Test queries that showcase MMR vs Similarity Search differences

queries = [
    # 1. Generic genre query - likely to return many similar superhero movies
    # Similarity search may return Marvel Avengers 1, 2, 3; MMR diversifies across DC, indie superhero films
    "superhero movies with action and special effects",
    
    # 2. Common director query - will find multiple films by same director
    # Similarity returns all Nolan films clustered together; MMR spreads across his different genres
    "movies directed by Christopher Nolan",
    
    # 3. Actor-focused query - tends to return sequels/franchises
    # Similarity might return Mission Impossible 1-6; MMR includes Top Gun, Edge of Tomorrow, etc.
    "films starring Tom Cruise in action roles",
    
    # 4. Theme-based query - attracts similar plotlines
    # Similarity clusters similar sci-fi; MMR includes comedy (Back to the Future), The Visitors
    "time travel science fiction movies",
    
    # 5. Keyword clustering query - groups similar concepts
    # Similarity returns similar psychological thrillers; MMR diversifies across different thriller subgenres
    "psychological thriller with plot twists and suspense"
]

# Test each query
for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    # Similarity Search - returns most similar, may be redundant
    similarity_results = faiss_db.similarity_search(query, k=3)
    print("\nSimilarity Search (k=3):")
    for i, doc in enumerate(similarity_results, 1):
        print(f"  {i}. {doc.metadata['title']}")
    
    # MMR - returns relevant but diverse results
    mmr_results = faiss_db.max_marginal_relevance_search(
        query, k=3, fetch_k=20, lambda_mult=0.5
    )
    print("\nMMR Results (k=3, fetch_k=20):")
    for i, doc in enumerate(mmr_results, 1):
        print(f"  {i}. {doc.metadata['title']}")


Query: superhero movies with action and special effects

Similarity Search (k=3):
  1. The Specials
  2. Superhero Movie
  3. Man of Steel

MMR Results (k=3, fetch_k=20):
  1. The Specials
  2. Man of Steel
  3. My Super Ex-Girlfriend

Query: movies directed by Christopher Nolan

Similarity Search (k=3):
  1. Insomnia
  2. The Dark Knight
  3. Noah

MMR Results (k=3, fetch_k=20):
  1. Insomnia
  2. Romeo + Juliet
  3. The Hobbit: An Unexpected Journey

Query: films starring Tom Cruise in action roles

Similarity Search (k=3):
  1. True Lies
  2. George and the Dragon
  3. Days of Thunder

MMR Results (k=3, fetch_k=20):
  1. True Lies
  2. Top Gun
  3. Warrior

Query: time travel science fiction movies

Similarity Search (k=3):
  1. The Time Machine
  2. Safety Not Guaranteed
  3. Time Changer

MMR Results (k=3, fetch_k=20):
  1. The Time Machine
  2. The Visitors
  3. Back to the Future Part II

Query: psychological thriller with plot twists and suspense

Similarity Search (k=3):
  1.

## Search Strategies

**1. For RAG/Q&A Systems (single best answer):**
````python
retriever = faiss_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
````
- Returns most relevant documents
- Best when you need precise matches

**2. For Discovery/Recommendation (diverse results):**
````python
retriever = faiss_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)
````
- Balances relevance and diversity
- Prevents showing only sequels/franchises
- Better user experience for exploration

**3. For Hybrid Precision + Diversity:**
````python
retriever = faiss_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.8,
        "k": 5
    }
)
````
- Only returns highly relevant results
- Filters out weak matches